# 05 SAJU ML V4 Feature / Model Tournament

**Notebook version:** `SAJU_ML_V4_FEATURE_MODEL_TOURNAMENT_20260816`

## Goal

The V4 target is now a **development target**, not a holdout. This notebook asks:

> After removing the obvious lifecycle imbalance from the old Career target, is there any astrology feature architecture that predicts same-axis positive-vs-negative Career years beyond non-astrology nuisance and the production Control?

## Target freeze

Required 04 decision:

`V4_TARGET_MODEL_READY`

Expected target:

- 100 explicit matched pairs
- 69 subjects
- subject-equal pair weights
- same-axis observed positive/negative events only
- no local non-event controls
- exact birth / modern engine lineage

## Important origin guard

The 100-pair target is balanced **overall**, but not inside its historical collection cohorts:

- prior 414-review pairs: positive earlier ≈ 83.7%
- chronology-expansion pairs: positive earlier = 0%

Therefore:

1. `pair_origin` is **forbidden** from all astrology models.
2. It is used only for robustness reporting.
3. A winner-eligible model must be at least chance (`>=0.50`) in **both** origins.
4. We also fit a forbidden-metadata diagnostic nuisance model (`age × origin`) only to quantify collection artifact risk; it can never win.

## Models

Baselines:

- fixed younger-year / later-year
- learned `AGE_AXIS_NO_ORIGIN`
- diagnostic `AGE_AXIS_PLUS_ORIGIN` (**forbidden predictive metadata**)
- production annual `candle.close` Control

Astrology candidates:

1. `TG5_GROUP_L2` — frozen-style 5 groups × Daewoon/Sewoon
2. `TG10_STEM_L2` — exact 10 Ten Gods on DW/SW stems
3. `TG10_STEM_BRANCH_L2` — exact stem + branch hidden-stem Ten-God distributions
4. `TG10_SB_STRENGTH_L2` — stem/branch Ten Gods + natal strength interactions
5. `TG10_SB_RELATIONS_L2` — stem/branch Ten Gods + 합충형파해 relation context
6. `ALL_L2` — full current feature universe
7. `ALL_ELASTICNET_AUTO` — nested automatic sparse selection
8. `HGB_ALL_EXPLORATORY` — nonlinear exploratory model; **not winner-eligible** at n=69 subjects

All tuning happens inside outer-training subjects only.

`NEW_CONFIRM`, Public CHECK/FINAL and Validation B are never loaded.

In [1]:
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
import hashlib, json, math, re, sys, warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HGB_AVAILABLE = True
except Exception:
    HGB_AVAILABLE = False

warnings.filterwarnings("ignore")

NOTEBOOK_VERSION = "SAJU_ML_V4_FEATURE_MODEL_TOURNAMENT_R2_20260816"
SEED = 20260816
OUTER_REPEATS = 5
OUTER_FOLDS = 5
INNER_FOLDS = 3
N_BOOTSTRAP = 20000

ELASTIC_C_GRID = [0.03, 0.10, 0.30, 1.00]
ELASTIC_L1_GRID = [0.25, 0.50, 0.75, 1.00]

# Predeclared discovery gate. Do not edit after seeing results.
MIN_OVERALL_MACRO = 0.55
MIN_DELTA_VS_REFERENCE = 0.01
MIN_BOOT_P_DELTA_GT_0 = 0.80
MIN_ORIGIN_MACRO = 0.50
MIN_P10 = 0.48


def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "saju_engine.py").exists():
            return candidate
    raise FileNotFoundError("Run inside the Chartpalja saju repo (saju_engine.py required).")

REPO_ROOT = find_repo_root()
TARGET_DIR = REPO_ROOT / "research/ml/artifacts/v4_model_ready_targets"
CORPUS_DIR = REPO_ROOT / "research/ml_corpus/v4_target_expansion"

PAIRS_PATH = TARGET_DIR / "V4_MODEL_READY_PAIRS.csv"
DECISION_PATH = TARGET_DIR / "V4_MODEL_READY_TARGET_DECISION.json"
CORPUS_PATH = CORPUS_DIR / "V4_TARGET_EXPANSION_CORPUS.json"
ORIGIN_GUARD_PATH = TARGET_DIR / "V4_TARGET_ORIGIN_GUARD_ADDENDUM.json"

ARTIFACT_DIR = REPO_ROOT / "research/ml/artifacts/v4_feature_model_tournament"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for p in [PAIRS_PATH, DECISION_PATH, CORPUS_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

print("="*90)
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)
print("repo:", REPO_ROOT)
print("output:", ARTIFACT_DIR)
print("="*90)

NOTEBOOK VERSION: SAJU_ML_V4_FEATURE_MODEL_TOURNAMENT_R2_20260816
repo: /Users/sangjinlee/Desktop/projects/saju
output: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_feature_model_tournament


## R2 CV splitter correction

This revision fixes a **CV allocation bug only**. No target, feature, model,
hyperparameter, or predeclared winner gate is changed.

The original greedy splitter minimized the candidate fold's **absolute**
post-assignment distance from its target. That can preferentially keep assigning
subjects to folds already near the target and leave other folds empty.

R2:

1. pre-seeds one subject into every fold;
2. minimizes **incremental imbalance** (`after_cost - before_cost`);
3. balances pair counts, axis counts, origin counts, and subject counts;
4. hard-fails on empty folds, train/test subject overlap, duplicate assignment,
   or incomplete coverage;
5. writes `V4_outer_fold_balance_R2.csv`.

This correction must be applied **before any tournament result is interpreted**.
The failed run produced no valid outer-CV tournament result.

## 1. Target and holdout preflight

In [2]:
pairs = pd.read_csv(PAIRS_PATH)
with open(DECISION_PATH, encoding="utf-8") as f:
    target_decision = json.load(f)
with open(CORPUS_PATH, encoding="utf-8") as f:
    corpus = json.load(f)

assert target_decision["status"] == "V4_TARGET_MODEL_READY"
assert len(pairs) == 100
assert pairs["subject_id"].nunique() == 69
assert pairs["model_ready"].astype(bool).all()
assert pairs["source_pair_eligible"].astype(bool).all()
assert (pairs["positive_year"] != pairs["negative_year"]).all()

weight_sums = pairs.groupby("subject_id")["subject_equal_pair_weight"].sum()
assert np.allclose(weight_sums.values, 1.0)

# Explicitly no sealed-set path exists in this notebook.
for text in [str(PAIRS_PATH), str(DECISION_PATH), str(CORPUS_PATH)]:
    upper = text.upper()
    assert "NEW_CONFIRM" not in upper
    assert "VALIDATION_B" not in upper
    assert "FINAL" not in upper
    assert "CHECK" not in upper

print("Target freeze: PASS")
print("pairs:", len(pairs), "subjects:", pairs.subject_id.nunique())
print(pairs.groupby("pair_origin").agg(
    pairs=("pair_id","size"),
    subjects=("subject_id","nunique"),
    positive_earlier=("positive_earlier_calc","mean")
))

Target freeze: PASS
pairs: 100 subjects: 69
                                pairs  subjects  positive_earlier
pair_origin                                                      
PREEXISTING_414_SOURCE_REVIEW      49        46          0.836735
V4_TARGET_EXPANSION_2026-08-16     51        37          0.000000


## 2. Mandatory origin-confounding diagnostic

This does not invalidate the target automatically. It changes the evaluation contract: overall performance alone is not sufficient.

In [3]:
origin_summary = (
    pairs.groupby("pair_origin")
    .agg(
        n_pairs=("pair_id", "size"),
        n_subjects=("subject_id", "nunique"),
        positive_earlier_share=("positive_earlier_calc", "mean"),
    )
    .reset_index()
)
origin_summary.to_csv(ARTIFACT_DIR / "V4_target_origin_summary.csv", index=False)
display(origin_summary)

assert len(origin_summary) == 2
# Guard, not a failure: target was intentionally constructed from opposing chronology cohorts.
ORIGIN_GUARD_REQUIRED = bool(
    origin_summary["positive_earlier_share"].max()
    - origin_summary["positive_earlier_share"].min() >= 0.50
)
print("ORIGIN_GUARD_REQUIRED:", ORIGIN_GUARD_REQUIRED)

,pair_origin,n_pairs,n_subjects,positive_earlier_share
0,PREEXISTING_414_SOURCE_REVIEW,49,46,0.836735
1,V4_TARGET_EXPANSION_2026-08-16,51,37,0.000000


ORIGIN_GUARD_REQUIRED: True


## 3. Canonical engine / astrology feature helpers

In [4]:
sys.path.insert(0, str(REPO_ROOT))

import saju_engine as se
import sajupy

STEMS = list("甲乙丙丁戊己庚辛壬癸")
BRANCHES = list("子丑寅卯辰巳午未申酉戌亥")
ELEMENTS = ["wood", "fire", "earth", "metal", "water"]

STEM_ELEMENT_EN = {
    "甲":"wood","乙":"wood","丙":"fire","丁":"fire","戊":"earth",
    "己":"earth","庚":"metal","辛":"metal","壬":"water","癸":"water"
}
STEM_YINYANG = {
    "甲":"yang","乙":"yin","丙":"yang","丁":"yin","戊":"yang",
    "己":"yin","庚":"yang","辛":"yin","壬":"yang","癸":"yin"
}
BRANCH_ELEMENT_EN = {
    "子":"water","丑":"earth","寅":"wood","卯":"wood","辰":"earth","巳":"fire",
    "午":"fire","未":"earth","申":"metal","酉":"metal","戌":"earth","亥":"water"
}
HIDDEN_STEMS = {
    "子":["癸"], "丑":["己","癸","辛"], "寅":["甲","丙","戊"], "卯":["乙"],
    "辰":["戊","乙","癸"], "巳":["丙","戊","庚"], "午":["丁","己"],
    "未":["己","丁","乙"], "申":["庚","壬","戊"], "酉":["辛"],
    "戌":["戊","辛","丁"], "亥":["壬","甲"]
}

GENERATES = {
    "wood":"fire","fire":"earth","earth":"metal","metal":"water","water":"wood"
}
CONTROLS = {
    "wood":"earth","earth":"water","water":"fire","fire":"metal","metal":"wood"
}

STEM_COMBINES = {
    frozenset(x)
    for x in [("甲","己"),("乙","庚"),("丙","辛"),("丁","壬"),("戊","癸")]
}
STEM_CLASHES = {
    frozenset(x)
    for x in [("甲","庚"),("乙","辛"),("丙","壬"),("丁","癸")]
}
BRANCH_COMBINES = {
    frozenset(x)
    for x in [("子","丑"),("寅","亥"),("卯","戌"),("辰","酉"),("巳","申"),("午","未")]
}
BRANCH_CLASHES = {
    frozenset(x)
    for x in [("子","午"),("丑","未"),("寅","申"),("卯","酉"),("辰","戌"),("巳","亥")]
}
BRANCH_HARMS = {
    frozenset(x)
    for x in [("子","未"),("丑","午"),("寅","巳"),("卯","辰"),("申","亥"),("酉","戌")]
}
BRANCH_BREAKS = {
    frozenset(x)
    for x in [("子","酉"),("丑","辰"),("寅","亥"),("卯","午"),("巳","申"),("未","戌")]
}
SELF_PUNISH = set(["辰","午","酉","亥"])
PUNISH_TRIPLES = [set("寅巳申"), set("丑未戌")]

UNSEONG_STATES = [
    "장생","목욕","관대","건록","제왕","쇠",
    "병","사","묘","절","태","양"
]
UNSEONG_START = {
    "甲":"亥", "乙":"午",
    "丙":"寅", "丁":"酉",
    "戊":"寅", "己":"酉",
    "庚":"巳", "辛":"子",
    "壬":"申", "癸":"卯",
}
YANG_STEMS = set("甲丙戊庚壬")
SHINSAL_KEYWORDS = ("신살", "귀인", "공망")

def parse_utc_offset(raw):
    if isinstance(raw, (int, float)):
        return float(raw)
    s = str(raw).strip()
    sign = -1.0 if s.startswith("-") else 1.0
    s = s[1:] if s[:1] in "+-" else s
    hh, mm = s.split(":")
    return sign * (int(hh) + int(mm) / 60.0)

def public_lon_corrected_birth(public_birth):
    y, m, d = map(int, public_birth["date"].split("-"))
    hh, mi = map(int, public_birth["time"].split(":")[:2])
    lon = float(public_birth["longitude"])
    utc_hours = parse_utc_offset(public_birth["utc_offset"])

    calc = sajupy.get_saju_calculator()
    civil = datetime(y, m, d, hh, mi)

    corr = float(
        calc._calculate_solar_time_correction(lon, utc_hours)
    )
    h2, min2, dc = calc._adjust_time_for_solar(
        civil.hour, civil.minute, corr
    )
    y2, m2, d2 = calc._adjust_date_for_solar(
        civil.year, civil.month, civil.day, dc
    )

    return {
        "calendar": public_birth.get("calendar", "solar"),
        "y": y2, "m": m2, "d": d2,
        "h": h2, "min": min2,
    }

def pillar_chars(value):
    if isinstance(value, (list, tuple)) and len(value) >= 2:
        a, b = str(value[0]), str(value[1])
        if a in STEMS and b in BRANCHES:
            return a, b

    s = str(value)
    stem = next((c for c in s if c in STEMS), None)
    branch = next((c for c in s if c in BRANCHES), None)
    return stem, branch

def sexagenary_year_pillar(year):
    i = (int(year) - 1984) % 60
    return STEMS[i % 10] + BRANCHES[i % 12]

def ten_god_group(day_stem, other_stem):
    de = STEM_ELEMENT_EN[day_stem]
    oe = STEM_ELEMENT_EN[other_stem]

    if de == oe:
        return "peer"
    if GENERATES[de] == oe:
        return "output"
    if CONTROLS[de] == oe:
        return "wealth"
    if CONTROLS[oe] == de:
        return "officer"
    if GENERATES[oe] == de:
        return "resource"

    raise RuntimeError((day_stem, other_stem))

def onehot(out, prefix, value, vocab):
    for v in vocab:
        out[prefix + str(v)] = float(value == v)

def pair_relation(a, b, relation_set):
    if a is None or b is None:
        return 0.0
    return float(frozenset((a, b)) in relation_set)

def branch_punishment(a, b):
    if a is None or b is None:
        return 0.0
    if a == b and a in SELF_PUNISH:
        return 1.0
    if set([a, b]) == set(["子", "卯"]):
        return 1.0
    return float(any(set([a, b]).issubset(x) for x in PUNISH_TRIPLES))

def get_strength_label(result):
    x = result.get("신강신약")
    if isinstance(x, dict):
        return x.get("판정") or x.get("label")
    if isinstance(x, str):
        return x
    return None

def get_daewoon_pillar(meta_row):
    for key in ("대운_pillar", "대운", "daewoon_pillar"):
        if meta_row.get(key):
            st, br = pillar_chars(meta_row[key])
            if st and br:
                return st + br
    return None

def get_daewoon_row(daewoon, year):
    for row in daewoon:
        if int(row["start_year"]) <= int(year) < int(row["end_year"]):
            return row
    return None

def get_sewoon_pillar(meta_row, year):
    for key in ("세운_pillar", "세운", "연주", "year_pillar"):
        if meta_row.get(key):
            st, br = pillar_chars(meta_row[key])
            if st and br:
                return st + br
    return sexagenary_year_pillar(year)

def control_score_from_meta(meta_row):
    candle = meta_row.get("candle") or {}
    value = candle.get("close")
    return float(value) if value is not None else np.nan

def local_twelve_unseong(day_stem, branch):
    start = UNSEONG_START[day_stem]
    start_i = BRANCHES.index(start)
    branch_i = BRANCHES.index(branch)
    direction = 1 if day_stem in YANG_STEMS else -1
    steps = ((branch_i - start_i) * direction) % 12
    return UNSEONG_STATES[steps]

def get_unseong_state(day_stem, branch):
    for fn_name in ("twelve_unseong", "_twelve_unseong"):
        fn = getattr(se, fn_name, None)
        if fn is None:
            continue

        for args in ((day_stem, branch), (branch, day_stem)):
            try:
                value = fn(*args)
                if isinstance(value, str) and value in UNSEONG_STATES:
                    return value
            except Exception:
                pass

    return local_twelve_unseong(day_stem, branch)

def strength_bucket(result):
    x = result.get("신강신약") or {}
    verdict = x.get("판정") if isinstance(x, dict) else str(x)
    verdict = verdict or ""

    if any(k in verdict for k in ("신강", "태강", "극왕")):
        return "strong"
    if any(k in verdict for k in ("신약", "태약", "극약")):
        return "weak"
    return "neutral"

# V1.3 Yongshin used the engine's element labels.
STEM_ELEMENT_ENGINE = getattr(se, "STEM_ELEMENT", {
    "甲":"목","乙":"목","丙":"화","丁":"화","戊":"토",
    "己":"토","庚":"금","辛":"금","壬":"수","癸":"수"
})
BRANCH_ELEMENT_ENGINE = getattr(se, "BRANCH_ELEMENT_MAIN", {
    "子":"수","丑":"토","寅":"목","卯":"목","辰":"토","巳":"화",
    "午":"화","未":"토","申":"금","酉":"금","戌":"토","亥":"수"
})

def yongshin_block_features(result, stem, branch, prefix):
    out = {}
    yong = result.get("용신") or {}
    day_stem = (result.get("원국") or {}).get("day", ["", ""])[0]

    fit = {}

    if yong and day_stem and hasattr(se, "_check_yongshin_fit"):
        try:
            fit = se._check_yongshin_fit(
                stem, branch, yong, day_stem
            ) or {}
        except Exception:
            fit = {}

    mapping = [
        ("용신부합", "yong_fit"),
        ("희신부합", "hee_fit"),
        ("기신부합", "gi_fit"),
        ("구신부합", "gu_fit"),
        ("용신부합_천간", "yong_stem_fit"),
        ("용신부합_지지", "yong_branch_fit"),
    ]

    for source_key, target_key in mapping:
        out[
            "yongshin__%s_%s" % (prefix, target_key)
        ] = float(fit.get(source_key) or 0.0)

    yong_e = yong.get("용신_오행") or ""
    hee = set(yong.get("희신_오행") or [])
    gi = set(yong.get("기신_오행") or [])
    gu = set(yong.get("구신_오행") or [])

    fav = (set([yong_e]) | hee) - set([""])
    unfav = (gi | gu) - set([""])

    stem_e = STEM_ELEMENT_ENGINE.get(stem, "")
    branch_e = BRANCH_ELEMENT_ENGINE.get(branch, "")

    out["yongshin__%s_supplies_fav" % prefix] = float(
        stem_e in fav or branch_e in fav
    )
    out["yongshin__%s_supplies_unfav" % prefix] = float(
        stem_e in unfav or branch_e in unfav
    )

    return out

def unseong_block_features(result, branch, prefix):
    out = {}
    day_stem = (result.get("원국") or {}).get("day", ["", ""])[0]
    state = get_unseong_state(day_stem, branch)

    for state_name in UNSEONG_STATES:
        out[
            "unseong__%s_state_%s" % (prefix, state_name)
        ] = float(state == state_name)

    raw_map = getattr(se, "_UNSEONG_SCORE", {})
    if isinstance(raw_map, dict):
        raw_score = float(raw_map.get(state, 0.0))
    else:
        raw_score = 0.0

    out["unseong__%s_raw_score" % prefix] = raw_score

    bucket = strength_bucket(result)
    sign = 1.0 if bucket == "weak" else (
        -1.0 if bucket == "strong" else 0.0
    )

    out["unseong__%s_score_x_strength" % prefix] = (
        raw_score * sign
    )

    return out

def safe_name(value):
    return re.sub(
        r"[^0-9A-Za-z가-힣_]+",
        "_",
        str(value),
    )[:120]

def flatten_all(value, path, out):
    key = safe_name(path)

    if isinstance(value, bool):
        out["shinsal__" + key] = float(value)

    elif isinstance(value, (int, float)) and np.isfinite(value):
        out["shinsal__" + key] = float(value)

    elif isinstance(value, str):
        if value.strip():
            out[
                "shinsal__%s__%s" % (key, safe_name(value))
            ] = 1.0

    elif isinstance(value, dict):
        for k2, v2 in value.items():
            flatten_all(v2, path + "." + str(k2), out)

    elif isinstance(value, (list, tuple, set)):
        out["shinsal__%s__count" % key] = float(len(value))

        for item in value:
            if isinstance(item, str):
                out[
                    "shinsal__%s__%s" % (key, safe_name(item))
                ] = 1.0
            elif isinstance(item, dict):
                flatten_all(item, path, out)

def flatten_selected(obj, prefix="", out=None):
    if out is None:
        out = {}

    if isinstance(obj, dict):
        for key, value in obj.items():
            path = prefix + "." + str(key) if prefix else str(key)

            if any(token in path for token in SHINSAL_KEYWORDS):
                flatten_all(value, path, out)

            elif isinstance(value, (dict, list, tuple)):
                flatten_selected(value, path, out)

    elif isinstance(obj, (list, tuple)):
        for i, value in enumerate(obj):
            flatten_selected(
                value,
                "%s[%d]" % (prefix, i),
                out,
            )

    return out

def shinsal_features(dw_row, meta_row):
    out = {}

    if dw_row:
        breakdown = dw_row.get("breakdown") or {}

        if "shinsal" in breakdown:
            out["shinsal__dw_breakdown_scalar"] = float(
                breakdown.get("shinsal") or 0.0
            )

        gongmang = dw_row.get("gongmang_factors") or {}

        for key, value in gongmang.items():
            if isinstance(value, (int, float)):
                out[
                    "shinsal__dw_gongmang_%s" % safe_name(key)
                ] = float(value)

        indicators = dw_row.get("indicators") or {}
        noble = indicators.get("귀인력")

        if isinstance(noble, (int, float)):
            out["shinsal__dw_noble_power"] = float(noble)

        flatten_selected(dw_row, "dw", out)

    flatten_selected(meta_row, "sw", out)
    return out

In [5]:
EXACT_TENGODS = [
    "bijian", "jiecai", "shishen", "shangguan",
    "pian_cai", "zheng_cai", "qi_sha", "zheng_guan",
    "pian_yin", "zheng_yin",
]

def exact_ten_god(day_stem, other_stem):
    de = STEM_ELEMENT_EN[day_stem]
    oe = STEM_ELEMENT_EN[other_stem]
    same_polarity = STEM_YINYANG[day_stem] == STEM_YINYANG[other_stem]

    if de == oe:
        return "bijian" if same_polarity else "jiecai"
    if GENERATES[de] == oe:
        return "shishen" if same_polarity else "shangguan"
    if CONTROLS[de] == oe:
        return "pian_cai" if same_polarity else "zheng_cai"
    if CONTROLS[oe] == de:
        return "qi_sha" if same_polarity else "zheng_guan"
    if GENERATES[oe] == de:
        return "pian_yin" if same_polarity else "zheng_yin"
    raise RuntimeError((day_stem, other_stem))

def exact_tg_distribution(day_stem, stems):
    counts = Counter(exact_ten_god(day_stem, st) for st in stems)
    n = float(max(1, len(stems)))
    return {tg: counts[tg] / n for tg in EXACT_TENGODS}

## 4. Current-runtime engine adapter for V4 subjects

In [6]:
import saju_engine as se
import sajupy

ENGINE_CALCULATOR = sajupy.get_saju_calculator()
ENGINE_MIN_YEAR = int(ENGINE_CALCULATOR.min_year)
ENGINE_MAX_YEAR = int(ENGINE_CALCULATOR.max_year)

subject_map = {s["subject_id"]: s for s in corpus["subjects"]}
missing_subjects = sorted(set(pairs.subject_id) - set(subject_map))
assert not missing_subjects, missing_subjects


def compute_v4_subject_engine(subject):
    birth = subject["birth"]
    corrected = public_lon_corrected_birth(birth)

    if not (ENGINE_MIN_YEAR <= int(corrected["y"]) <= ENGINE_MAX_YEAR):
        raise ValueError((subject["subject_id"], corrected["y"], ENGINE_MIN_YEAR, ENGINE_MAX_YEAR))

    inp = se.BirthInput(
        year=int(corrected["y"]), month=int(corrected["m"]), day=int(corrected["d"]),
        hour=int(corrected["h"]), minute=int(corrected["min"]),
        gender=subject["gender"], calendar=corrected.get("calendar", "solar"),
        is_leap_month=False, use_solar_time=False, utc_offset=9,
    )

    result = se.compute_all(inp)
    daewoon = se.build_daewoon_detail(result)
    timeline = result["chart_data"]["연도별_타임라인"]
    meta = {int(row["year"]): row for row in timeline}
    return result, daewoon, meta

## 5. Full feature extractor + exact Ten-God extensions

In [7]:
def extract_v2_all_features(subject, year, result, daewoon, meta_row):
    natal = result["원국"]

    natal_pillars = {}
    for pos in ("year", "month", "day", "hour"):
        stem, branch = pillar_chars(natal[pos])
        natal_pillars[pos] = (stem, branch)

    day_stem = natal_pillars["day"][0]

    # Preserve V1 base feature semantics by reading the Daewoon pillar
    # from the annual timeline metadata.
    dw_stem, dw_branch = pillar_chars(
        get_daewoon_pillar(meta_row)
    )

    sw_stem, sw_branch = pillar_chars(
        get_sewoon_pillar(meta_row, year)
    )

    if not all([
        day_stem,
        dw_stem,
        dw_branch,
        sw_stem,
        sw_branch,
    ]):
        raise ValueError(
            "Pillar parse failed: %s %s"
            % (subject["subject_id"], year)
        )

    features = {}

    # BASIC
    for pos, pair in natal_pillars.items():
        stem, branch = pair
        onehot(
            features,
            "basic__natal_%s_stem_" % pos,
            stem,
            STEMS,
        )
        onehot(
            features,
            "basic__natal_%s_branch_" % pos,
            branch,
            BRANCHES,
        )

    onehot(
        features,
        "basic__daymaster_stem_",
        day_stem,
        STEMS,
    )
    onehot(
        features,
        "basic__daymaster_element_",
        STEM_ELEMENT_EN[day_stem],
        ELEMENTS,
    )

    for label, stem, branch in [
        ("dw", dw_stem, dw_branch),
        ("sw", sw_stem, sw_branch),
    ]:
        onehot(
            features,
            "basic__%s_stem_" % label,
            stem,
            STEMS,
        )
        onehot(
            features,
            "basic__%s_branch_" % label,
            branch,
            BRANCHES,
        )
        onehot(
            features,
            "basic__%s_element_" % label,
            STEM_ELEMENT_EN[stem],
            ELEMENTS,
        )
        onehot(
            features,
            "basic__%s_yy_" % label,
            STEM_YINYANG[stem],
            ["yang", "yin"],
        )

    natal_element_counts = Counter()

    for stem, branch in natal_pillars.values():
        natal_element_counts[STEM_ELEMENT_EN[stem]] += 1
        natal_element_counts[BRANCH_ELEMENT_EN[branch]] += 1

    for element in ELEMENTS:
        features[
            "basic__natal_element_count_%s" % element
        ] = float(natal_element_counts[element])

    # TENGOD
    for label, stem in [
        ("dw", dw_stem),
        ("sw", sw_stem),
    ]:
        tg = ten_god_group(day_stem, stem)
        onehot(
            features,
            "tengod__%s_group_" % label,
            tg,
            ["peer", "output", "wealth", "officer", "resource"],
        )

    # HIDDEN
    hidden_tg = Counter()
    hidden_element = Counter()

    for _, branch in natal_pillars.values():
        for hidden_stem in HIDDEN_STEMS[branch]:
            hidden_tg[
                ten_god_group(day_stem, hidden_stem)
            ] += 1

            hidden_element[
                STEM_ELEMENT_EN[hidden_stem]
            ] += 1

    for tg in [
        "peer", "output", "wealth", "officer", "resource"
    ]:
        features[
            "hidden__natal_tg_count_%s" % tg
        ] = float(hidden_tg[tg])

    for element in ELEMENTS:
        features[
            "hidden__natal_element_count_%s" % element
        ] = float(hidden_element[element])

    # RELATIONS
    natal_stems = [
        pair[0] for pair in natal_pillars.values()
    ]
    natal_branches = [
        pair[1] for pair in natal_pillars.values()
    ]

    for label, stem, branch in [
        ("dw", dw_stem, dw_branch),
        ("sw", sw_stem, sw_branch),
    ]:
        features[
            "relation__%s_stem_combine_natal" % label
        ] = sum(
            pair_relation(stem, x, STEM_COMBINES)
            for x in natal_stems
        )

        features[
            "relation__%s_stem_clash_natal" % label
        ] = sum(
            pair_relation(stem, x, STEM_CLASHES)
            for x in natal_stems
        )

        features[
            "relation__%s_branch_combine_natal" % label
        ] = sum(
            pair_relation(branch, x, BRANCH_COMBINES)
            for x in natal_branches
        )

        features[
            "relation__%s_branch_clash_natal" % label
        ] = sum(
            pair_relation(branch, x, BRANCH_CLASHES)
            for x in natal_branches
        )

        features[
            "relation__%s_branch_harm_natal" % label
        ] = sum(
            pair_relation(branch, x, BRANCH_HARMS)
            for x in natal_branches
        )

        features[
            "relation__%s_branch_break_natal" % label
        ] = sum(
            pair_relation(branch, x, BRANCH_BREAKS)
            for x in natal_branches
        )

        features[
            "relation__%s_branch_punish_natal" % label
        ] = sum(
            branch_punishment(branch, x)
            for x in natal_branches
        )

    features["relation__sw_dw_stem_combine"] = pair_relation(
        sw_stem, dw_stem, STEM_COMBINES
    )
    features["relation__sw_dw_stem_clash"] = pair_relation(
        sw_stem, dw_stem, STEM_CLASHES
    )
    features["relation__sw_dw_branch_combine"] = pair_relation(
        sw_branch, dw_branch, BRANCH_COMBINES
    )
    features["relation__sw_dw_branch_clash"] = pair_relation(
        sw_branch, dw_branch, BRANCH_CLASHES
    )
    features["relation__sw_dw_branch_harm"] = pair_relation(
        sw_branch, dw_branch, BRANCH_HARMS
    )
    features["relation__sw_dw_branch_break"] = pair_relation(
        sw_branch, dw_branch, BRANCH_BREAKS
    )
    features["relation__sw_dw_branch_punish"] = branch_punishment(
        sw_branch, dw_branch
    )

    # ENGINE STRENGTH
    strength = get_strength_label(result)

    for value in [
        "극약","태약","신약","중화신약",
        "중화신강","신강","태강","극왕"
    ]:
        features[
            "engine__strength_%s" % value
        ] = float(strength == value)

    # V1.3 EXTENDED
    dw_row = get_daewoon_row(daewoon, year)

    if dw_row is None:
        raise ValueError(
            "No active Daewoon row: %s %s"
            % (subject["subject_id"], year)
        )

    features.update(
        yongshin_block_features(
            result, dw_stem, dw_branch, "dw"
        )
    )
    features.update(
        yongshin_block_features(
            result, sw_stem, sw_branch, "sw"
        )
    )

    features.update(
        unseong_block_features(
            result, dw_branch, "dw"
        )
    )
    features.update(
        unseong_block_features(
            result, sw_branch, "sw"
        )
    )

    features.update(
        shinsal_features(dw_row, meta_row)
    )

    features["baseline__control_score"] = (
        control_score_from_meta(meta_row)
    )

    return features

In [8]:
def extract_v4_features(subject, year, result, daewoon, meta_row):
    f = extract_v2_all_features(subject, year, result, daewoon, meta_row)

    natal = result["원국"]
    day_stem, _ = pillar_chars(natal["day"])
    dw_stem, dw_branch = pillar_chars(get_daewoon_pillar(meta_row))
    sw_stem, sw_branch = pillar_chars(get_sewoon_pillar(meta_row, year))

    strength = strength_bucket(result)

    for label, stem, branch in [("dw", dw_stem, dw_branch), ("sw", sw_stem, sw_branch)]:
        tg = exact_ten_god(day_stem, stem)
        onehot(f, "tg10__%s_stem_" % label, tg, EXACT_TENGODS)

        dist = exact_tg_distribution(day_stem, HIDDEN_STEMS[branch])
        for exact_name in EXACT_TENGODS:
            f["tg10__%s_branch_hidden_%s" % (label, exact_name)] = float(dist[exact_name])

        # Contextual interaction: exact dynamic Ten-God activation x natal strength bucket.
        dynamic = {k: 0.5 * (float(tg == k) + float(dist[k])) for k in EXACT_TENGODS}
        for exact_name in EXACT_TENGODS:
            for bucket in ["weak", "neutral", "strong"]:
                f["tgxstrength__%s_%s_%s" % (label, exact_name, bucket)] = (
                    dynamic[exact_name] * float(strength == bucket)
                )

    return f

## 6. Build/cache year-level features for the 100 explicit pairs

In [9]:
FEATURE_CACHE = ARTIFACT_DIR / "V4_model_ready_year_features.csv"

required_years = defaultdict(set)
for _, r in pairs.iterrows():
    required_years[r.subject_id].add(int(r.positive_year))
    required_years[r.subject_id].add(int(r.negative_year))

rows = []
engine_cache = {}

for i, sid in enumerate(sorted(required_years)):
    subject = subject_map[sid]
    result, daewoon, meta = compute_v4_subject_engine(subject)
    engine_cache[sid] = (result, daewoon, meta)

    for year in sorted(required_years[sid]):
        if year not in meta:
            raise RuntimeError("Timeline missing %s %s" % (sid, year))
        features = extract_v4_features(subject, year, result, daewoon, meta[year])
        row = {"subject_id": sid, "year": int(year)}
        row.update(features)
        rows.append(row)

    if (i + 1) % 10 == 0:
        print("engine subjects", i + 1, "/", len(required_years))

year_features = pd.DataFrame(rows)
year_features.to_csv(FEATURE_CACHE, index=False)
print("year feature rows:", len(year_features), "columns:", len(year_features.columns))

[SAJU_DEBUG] original_input: 1968-03-12 18:41
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1968-03-12
[SAJU_DEBUG] final_datetime(KST): 1968-03-12T18:41:00+09:00
[SAJU_DEBUG] pillars: 연=戊申 월=乙卯 일=辛巳 시=丁酉
[SAJU_DEBUG] original_input: 1969-02-11 22:29
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1969-02-11
[SAJU_DEBUG] final_datetime(KST): 1969-02-11T22:29:00+09:00
[SAJU_DEBUG] pillars: 연=己酉 월=丙寅 일=丁巳 시=辛亥
[SAJU_DEBUG] original_input: 1967-10-27 22:38
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1967-10-27
[SAJU_DEBUG] final_datetime(KST): 1967-10-27T22:38:00+09:00
[SAJU_DEBUG] pillars: 연=丁未 월=庚戌 일=甲子 시=乙亥
[SAJU_DEBUG] original_input: 1969-12-21 21:49
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=femal

engine subjects 10 / 69


[SAJU_DEBUG] original_input: 1947-09-06 13:14
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1947-09-06
[SAJU_DEBUG] final_datetime(KST): 1947-09-06T13:14:00+09:00
[SAJU_DEBUG] 반시보정: 己未→戊午 (13:14)
[SAJU_DEBUG] pillars: 연=丁亥 월=戊申 일=戊子 시=戊午
[SAJU_DEBUG] original_input: 1960-03-26 11:49
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1960-03-26
[SAJU_DEBUG] final_datetime(KST): 1960-03-26T11:49:00+09:00
[SAJU_DEBUG] pillars: 연=庚子 월=己卯 일=癸丑 시=戊午
[SAJU_DEBUG] original_input: 1955-01-18 21:47
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1955-01-18
[SAJU_DEBUG] final_datetime(KST): 1955-01-18T21:47:00+09:00
[SAJU_DEBUG] pillars: 연=甲午 월=丁丑 일=己卯 시=乙亥
[SAJU_DEBUG] original_input: 1967-02-10 07:54
[SAJU_DEBUG] calendar=solar, is

engine subjects 20 / 69


[SAJU_DEBUG] original_input: 1949-10-08 18:18
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1949-10-08
[SAJU_DEBUG] final_datetime(KST): 1949-10-08T18:18:00+09:00
[SAJU_DEBUG] pillars: 연=己丑 월=癸酉 일=辛未 시=丁酉
[SAJU_DEBUG] original_input: 1945-08-14 04:25
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1945-08-14
[SAJU_DEBUG] final_datetime(KST): 1945-08-14T04:25:00+09:00
[SAJU_DEBUG] pillars: 연=乙酉 월=甲申 일=乙卯 시=戊寅
[SAJU_DEBUG] original_input: 1970-04-29 13:06
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1970-04-29
[SAJU_DEBUG] final_datetime(KST): 1970-04-29T13:06:00+09:00
[SAJU_DEBUG] 반시보정: 辛未→庚午 (13:06)
[SAJU_DEBUG] pillars: 연=庚戌 월=庚辰 일=己卯 시=庚午
[SAJU_DEBUG] original_input: 1946-01-05 02:56
[SAJU_DEBUG] calendar=solar, 

...

engine subjects 50 / 69


[SAJU_DEBUG] original_input: 1982-05-17 14:52
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1982-05-17
[SAJU_DEBUG] final_datetime(KST): 1982-05-17T14:52:00+09:00
[SAJU_DEBUG] pillars: 연=壬戌 월=乙巳 일=庚子 시=癸未
[SAJU_DEBUG] original_input: 1949-01-10 20:57
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1949-01-10
[SAJU_DEBUG] final_datetime(KST): 1949-01-10T20:57:00+09:00
[SAJU_DEBUG] pillars: 연=戊子 월=乙丑 일=庚子 시=丙戌
[SAJU_DEBUG] original_input: 1954-08-16 04:48
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1954-08-16
[SAJU_DEBUG] final_datetime(KST): 1954-08-16T04:48:00+09:00
[SAJU_DEBUG] pillars: 연=甲午 월=壬申 일=甲辰 시=丙寅
[SAJU_DEBUG] original_input: 1913-07-14 00:19
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

engine subjects 60 / 69


[SAJU_DEBUG] original_input: 1974-02-10 19:17
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1974-02-10
[SAJU_DEBUG] final_datetime(KST): 1974-02-10T19:17:00+09:00
[SAJU_DEBUG] 반시보정: 庚戌→己酉 (19:17)
[SAJU_DEBUG] pillars: 연=甲寅 월=丙寅 일=壬午 시=己酉
[SAJU_DEBUG] original_input: 1936-07-29 00:33
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1936-07-29
[SAJU_DEBUG] final_datetime(KST): 1936-07-29T00:33:00+09:00
[SAJU_DEBUG] pillars: 연=丙子 월=乙未 일=壬子 시=庚子
[SAJU_DEBUG] original_input: 1989-02-16 18:56
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1989-02-16
[SAJU_DEBUG] final_datetime(KST): 1989-02-16T18:56:00+09:00
[SAJU_DEBUG] pillars: 연=己巳 월=丙寅 일=丁未 시=己酉
[SAJU_DEBUG] original_input: 1917-11-20 13:35
[SAJU_DEBUG] calendar=solar

year feature rows: 177 columns: 405


## 7. Convert the explicit target pairs into pair-difference features

In [10]:
meta_cols = {"subject_id", "year"}
feature_cols = [c for c in year_features.columns if c not in meta_cols]
for c in feature_cols:
    year_features[c] = pd.to_numeric(year_features[c], errors="coerce").fillna(0.0)

lookup = year_features.set_index(["subject_id", "year"])
pair_rows = []

for _, r in pairs.iterrows():
    p = lookup.loc[(r.subject_id, int(r.positive_year))]
    n = lookup.loc[(r.subject_id, int(r.negative_year))]

    out = r.to_dict()
    for c in feature_cols:
        out["diff__" + c] = float(p[c] - n[c])

    # Nuisance features: no collection origin in winner-eligible nuisance model.
    age_delta = float(r.positive_age - r.negative_age)
    age_mid = 0.5 * float(r.positive_age + r.negative_age)
    out["nuisance__age_delta"] = age_delta
    out["nuisance__age_delta_x_gap"] = age_delta * float(r.abs_year_gap)
    out["nuisance__age_delta_x_mid"] = age_delta * age_mid

    for axis in sorted(pairs.axis.unique()):
        out["nuisance__age_delta_x_axis__" + axis] = age_delta * float(r.axis == axis)

    # Forbidden metadata diagnostic only.
    for origin in sorted(pairs.pair_origin.unique()):
        out["forbidden__age_delta_x_origin__" + re.sub(r"[^A-Za-z0-9_]+", "_", origin)] = (
            age_delta * float(r.pair_origin == origin)
        )

    pair_rows.append(out)

pair_df = pd.DataFrame(pair_rows)
pair_df.to_csv(ARTIFACT_DIR / "V4_pair_diff_feature_table.csv", index=False)
print("pair diff table:", pair_df.shape)

pair diff table: (100, 443)


## 8. Feature families

In [11]:
ALL_DIFF = [c for c in pair_df.columns if c.startswith("diff__")]

def select(prefix):
    return sorted([c for c in ALL_DIFF if c.startswith("diff__" + prefix)])

TG5 = select("tengod__")
TG10_STEM = sorted([c for c in ALL_DIFF if c.startswith("diff__tg10__") and "_stem_" in c])
TG10_BRANCH = sorted([c for c in ALL_DIFF if c.startswith("diff__tg10__") and "_branch_hidden_" in c])
TG10_SB = sorted(set(TG10_STEM + TG10_BRANCH))
TGX_STRENGTH = select("tgxstrength__")
RELATIONS = select("relation__")

# Full universe excludes production Control and the exact duplicate metadata baseline.
ALL_ASTRO = sorted([
    c for c in ALL_DIFF
    if not c.startswith("diff__baseline__")
])

NUISANCE_NO_ORIGIN = sorted([c for c in pair_df.columns if c.startswith("nuisance__")])
FORBIDDEN_ORIGIN = sorted([c for c in pair_df.columns if c.startswith("forbidden__")])

FEATURE_SETS = {
    "TG5_GROUP_L2": TG5,
    "TG10_STEM_L2": TG10_STEM,
    "TG10_STEM_BRANCH_L2": TG10_SB,
    "TG10_SB_STRENGTH_L2": sorted(set(TG10_SB + TGX_STRENGTH)),
    "TG10_SB_RELATIONS_L2": sorted(set(TG10_SB + RELATIONS)),
    "ALL_L2": ALL_ASTRO,
}

for name, cols in FEATURE_SETS.items():
    if not cols:
        raise RuntimeError("Empty feature set: %s" % name)
    print("%-28s %4d" % (name, len(cols)))
print("ALL_ASTRO", len(ALL_ASTRO))
print("NUISANCE_NO_ORIGIN", len(NUISANCE_NO_ORIGIN))

TG5_GROUP_L2                   10
TG10_STEM_L2                   20
TG10_STEM_BRANCH_L2            40
TG10_SB_STRENGTH_L2           100
TG10_SB_RELATIONS_L2           61
ALL_L2                        402
ALL_ASTRO 402
NUISANCE_NO_ORIGIN 7


## 9. Pairwise model utilities with subject-equal training weight

In [12]:
def symmetric_xy(frame, columns):
    X = frame[columns].to_numpy(dtype=float)
    y = np.r_[np.ones(len(X), dtype=int), np.zeros(len(X), dtype=int)]
    X2 = np.vstack([X, -X])
    w0 = frame["subject_equal_pair_weight"].to_numpy(dtype=float) / 2.0
    w = np.r_[w0, w0]
    return X2, y, w


def fit_l2(frame, columns, C=0.3):
    X, y, w = symmetric_xy(frame, columns)
    model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2", C=float(C), solver="liblinear", max_iter=5000,
            random_state=SEED,
        )),
    ])
    model.fit(X, y, model__sample_weight=w)
    return model


def fit_elastic(frame, columns, C, l1_ratio, seed):
    X, y, w = symmetric_xy(frame, columns)
    model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            penalty="elasticnet", solver="saga", C=float(C), l1_ratio=float(l1_ratio),
            max_iter=8000, random_state=int(seed),
        )),
    ])
    model.fit(X, y, model__sample_weight=w)
    return model


def fit_hgb(frame, columns, seed):
    X, y, w = symmetric_xy(frame, columns)
    if HGB_AVAILABLE:
        model = HistGradientBoostingClassifier(
            learning_rate=0.05, max_iter=180, max_leaf_nodes=7, max_depth=3,
            min_samples_leaf=10, l2_regularization=1.0, random_state=int(seed),
        )
    else:
        model = GradientBoostingClassifier(
            n_estimators=120, learning_rate=0.04, max_depth=2,
            min_samples_leaf=8, random_state=int(seed),
        )
    model.fit(X, y, sample_weight=w)
    return model


def decision_scores(model, frame, columns):
    X = frame[columns].to_numpy(dtype=float)
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(X), dtype=float)
    proba = model.predict_proba(X)[:, 1]
    return np.asarray(proba - 0.5, dtype=float)


def pair_correct_from_score(score, eps=1e-12):
    score = np.asarray(score, dtype=float)
    return np.where(score > eps, 1.0, np.where(score < -eps, 0.0, 0.5))


def subject_macro(frame, correct):
    tmp = frame[["subject_id"]].copy()
    tmp["correct"] = np.asarray(correct, dtype=float)
    return float(tmp.groupby("subject_id")["correct"].mean().mean())


def per_subject(frame, correct):
    tmp = frame[["subject_id"]].copy()
    tmp["correct"] = np.asarray(correct, dtype=float)
    return tmp.groupby("subject_id")["correct"].mean()

## 10. Subject-disjoint balanced folds

In [13]:
def make_subject_folds(frame, n_folds, seed):
    """
    Subject-disjoint greedy fold allocator.

    R2 FIX:
    The V1 splitter minimized the *absolute post-assignment distance* of only
    the candidate fold from its target. A fold already close to target could
    therefore keep winning, leaving other folds empty.

    R2 instead minimizes the *incremental change in global imbalance* for the
    candidate fold (after_cost - before_cost), pre-seeds one subject per fold,
    and hard-checks coverage/disjointness/non-empty folds.
    """
    rng = np.random.RandomState(int(seed))
    subject_rows = []
    axes = sorted(frame.axis.unique())
    origins = sorted(frame.pair_origin.unique())

    for sid, g in frame.groupby("subject_id"):
        row = {"subject_id": sid, "n": len(g)}
        for x in axes:
            row["axis__" + x] = int((g.axis == x).sum())
        for x in origins:
            row["origin__" + x] = int((g.pair_origin == x).sum())
        subject_rows.append(row)

    meta = pd.DataFrame(subject_rows)
    n_subjects = int(len(meta))
    n_folds = int(min(int(n_folds), n_subjects))
    if n_folds < 2:
        raise ValueError("Need at least 2 unique subjects for subject-CV.")

    meta["jitter"] = rng.uniform(size=n_subjects)
    meta = meta.sort_values(
        ["n", "jitter"], ascending=[False, True]
    ).reset_index(drop=True)

    count_cols = [c for c in meta.columns if c not in {"subject_id", "jitter"}]
    totals = meta[count_cols].sum().to_numpy(dtype=float)
    targets = totals / float(n_folds)
    scale = np.maximum(targets, 1.0)

    fold_sums = np.zeros((n_folds, len(count_cols)), dtype=float)
    folds = [[] for _ in range(n_folds)]
    subject_target = n_subjects / float(n_folds)

    # Guarantee non-empty folds before the greedy phase.
    for k in range(n_folds):
        r = meta.iloc[k]
        vec = r[count_cols].to_numpy(dtype=float)
        folds[k].append(r.subject_id)
        fold_sums[k] += vec

    # Greedy assignment by INCREMENTAL imbalance, not absolute fold cost.
    for idx in range(n_folds, n_subjects):
        r = meta.iloc[idx]
        vec = r[count_cols].to_numpy(dtype=float)
        costs = []

        for k in range(n_folds):
            before = float(np.sum(((fold_sums[k] - targets) / scale) ** 2))
            after = float(np.sum(((fold_sums[k] + vec - targets) / scale) ** 2))

            # Small subject-count balance term. Pair-count balance is already
            # present through the "n" feature above.
            before_n = ((len(folds[k]) - subject_target) / max(subject_target, 1.0)) ** 2
            after_n = ((len(folds[k]) + 1 - subject_target) / max(subject_target, 1.0)) ** 2

            delta_cost = (after - before) + 0.10 * (after_n - before_n)

            # Deterministic random tie breaker.
            delta_cost += 1e-9 * float(rng.uniform())
            costs.append(delta_cost)

        best = int(np.argmin(costs))
        folds[best].append(r.subject_id)
        fold_sums[best] += vec

    # Hard integrity checks.
    assert len(folds) == n_folds
    assert all(len(fold) > 0 for fold in folds), "Empty subject-CV fold detected."

    assigned = [sid for fold in folds for sid in fold]
    expected = set(meta.subject_id.astype(str))
    observed = set(map(str, assigned))

    assert len(assigned) == n_subjects, "Subject assigned more/less than once."
    assert len(observed) == n_subjects, "Duplicate subject assignment across folds."
    assert observed == expected, "Fold union does not equal the full subject set."

    return folds


def assert_subject_folds(frame, folds, label):
    """Fail early before any sklearn model sees an empty train/test matrix."""
    expected_subjects = set(frame.subject_id.astype(str))
    seen = []

    assert len(folds) >= 2, "%s: need >=2 folds" % label

    for k, fold in enumerate(folds):
        assert len(fold) > 0, "%s: fold %d has zero subjects" % (label, k)
        seen.extend(map(str, fold))

        te = frame[frame.subject_id.astype(str).isin(set(map(str, fold)))]
        tr = frame[~frame.subject_id.astype(str).isin(set(map(str, fold)))]

        assert not te.empty, "%s: fold %d has zero test rows" % (label, k)
        assert not tr.empty, "%s: fold %d has zero train rows" % (label, k)
        assert set(tr.subject_id.astype(str)).isdisjoint(
            set(te.subject_id.astype(str))
        ), "%s: train/test subject leakage in fold %d" % (label, k)

    assert len(seen) == len(expected_subjects), "%s: subject assignment count mismatch" % label
    assert len(set(seen)) == len(expected_subjects), "%s: duplicate subject across folds" % label
    assert set(seen) == expected_subjects, "%s: fold coverage mismatch" % label

    return True

## 11. Nested ElasticNet tuning

In [14]:
def tune_elastic(train_frame, columns, seed):
    folds = make_subject_folds(
        train_frame,
        min(INNER_FOLDS, train_frame.subject_id.nunique()),
        seed,
    )
    assert_subject_folds(train_frame, folds, "INNER_CV")
    rows = []

    for C in ELASTIC_C_GRID:
        for l1 in ELASTIC_L1_GRID:
            scores = []
            for fold_i, test_subjects in enumerate(folds):
                tr = train_frame[~train_frame.subject_id.isin(test_subjects)]
                te = train_frame[train_frame.subject_id.isin(test_subjects)]
                if tr.subject_id.nunique() < 5 or te.empty:
                    continue
                model = fit_elastic(tr, columns, C, l1, seed + fold_i)
                correct = pair_correct_from_score(decision_scores(model, te, columns))
                scores.append(subject_macro(te, correct))
            rows.append({
                "C": C, "l1_ratio": l1,
                "inner_macro": float(np.mean(scores)) if scores else np.nan,
                "inner_p10": float(np.quantile(scores, .10)) if scores else np.nan,
            })

    grid = pd.DataFrame(rows).sort_values(
        ["inner_macro", "inner_p10", "C"], ascending=[False, False, True]
    ).reset_index(drop=True)
    return grid.iloc[0].to_dict(), grid

## 12. Run repeated nested subject-CV tournament

In [15]:
ASTRO_MODELS = list(FEATURE_SETS.keys()) + ["ALL_ELASTICNET_AUTO", "HGB_ALL_EXPLORATORY"]
ALL_MODELS = [
    "AGE_YOUNGER_FIXED", "AGE_LATER_FIXED", "NUISANCE_AGE_AXIS",
    "NUISANCE_WITH_ORIGIN_DIAGNOSTIC", "CONTROL_FIXED",
] + ASTRO_MODELS

outer_pair_rows = []
outer_subject_rows = []
elastic_grid_rows = []
elastic_selection_rows = []
outer_fold_balance_rows = []

# Production control pair correctness is fixed from year feature delta.
CONTROL_COL = "diff__baseline__control_score"
assert CONTROL_COL in pair_df.columns

for repeat in range(OUTER_REPEATS):
    folds = make_subject_folds(pair_df, OUTER_FOLDS, SEED + 1000 * repeat)
    assert_subject_folds(pair_df, folds, "OUTER_CV_REPEAT_%d" % repeat)

    for fold_i, test_subjects in enumerate(folds):
        train = pair_df[~pair_df.subject_id.isin(test_subjects)].copy()
        test = pair_df[pair_df.subject_id.isin(test_subjects)].copy()

        assert len(test_subjects) > 0
        assert not train.empty
        assert not test.empty
        assert set(train.subject_id).isdisjoint(set(test.subject_id))

        diag = {
            "repeat": repeat,
            "fold": fold_i,
            "n_test_subjects": int(test.subject_id.nunique()),
            "n_test_pairs": int(len(test)),
        }
        for origin in sorted(pair_df.pair_origin.unique()):
            diag["origin__" + re.sub(r"[^A-Za-z0-9_]+", "_", str(origin))] = int(
                (test.pair_origin == origin).sum()
            )
        for axis in sorted(pair_df.axis.unique()):
            diag["axis__" + str(axis)] = int((test.axis == axis).sum())
        outer_fold_balance_rows.append(diag)

        models = {}
        for name, cols in FEATURE_SETS.items():
            models[name] = (fit_l2(train, cols, C=0.3), cols)

        nuisance = fit_l2(train, NUISANCE_NO_ORIGIN, C=0.3)
        models["NUISANCE_AGE_AXIS"] = (nuisance, NUISANCE_NO_ORIGIN)

        forbidden_cols = sorted(set(NUISANCE_NO_ORIGIN + FORBIDDEN_ORIGIN))
        forbidden = fit_l2(train, forbidden_cols, C=0.3)
        models["NUISANCE_WITH_ORIGIN_DIAGNOSTIC"] = (forbidden, forbidden_cols)

        best_elastic, grid = tune_elastic(train, ALL_ASTRO, SEED + 10000*repeat + fold_i)
        grid["repeat"] = repeat
        grid["fold"] = fold_i
        elastic_grid_rows.append(grid)
        elastic_model = fit_elastic(
            train, ALL_ASTRO, best_elastic["C"], best_elastic["l1_ratio"],
            SEED + 20000*repeat + fold_i,
        )
        models["ALL_ELASTICNET_AUTO"] = (elastic_model, ALL_ASTRO)

        coef = elastic_model.named_steps["model"].coef_.ravel()
        for col, value in zip(ALL_ASTRO, coef):
            elastic_selection_rows.append({
                "repeat":repeat,"fold":fold_i,"feature":col,
                "selected":float(abs(value)>1e-8),"coefficient":float(value),
            })

        hgb = fit_hgb(train, ALL_ASTRO, SEED + 30000*repeat + fold_i)
        models["HGB_ALL_EXPLORATORY"] = (hgb, ALL_ASTRO)

        fixed = {
            "AGE_YOUNGER_FIXED": test["positive_earlier_calc"].to_numpy(dtype=float),
            "AGE_LATER_FIXED": 1.0 - test["positive_earlier_calc"].to_numpy(dtype=float),
            "CONTROL_FIXED": pair_correct_from_score(test[CONTROL_COL].to_numpy(dtype=float)),
        }

        scored = {}
        for name, correct in fixed.items():
            scored[name] = np.asarray(correct, dtype=float)
        for name, (model, cols) in models.items():
            scored[name] = pair_correct_from_score(decision_scores(model, test, cols))

        for name, correct in scored.items():
            tmp = test[["pair_id","subject_id","pair_origin","axis"]].copy()
            tmp["repeat"] = repeat
            tmp["fold"] = fold_i
            tmp["model"] = name
            tmp["correct"] = correct
            outer_pair_rows.append(tmp)

            sm = tmp.groupby("subject_id")["correct"].mean().reset_index()
            sm["repeat"] = repeat
            sm["fold"] = fold_i
            sm["model"] = name
            outer_subject_rows.append(sm)

outer_pair = pd.concat(outer_pair_rows, ignore_index=True)
outer_subject = pd.concat(outer_subject_rows, ignore_index=True)
elastic_grid = pd.concat(elastic_grid_rows, ignore_index=True)
elastic_selection = pd.DataFrame(elastic_selection_rows)

outer_pair.to_csv(ARTIFACT_DIR / "V4_OOF_pair_scores.csv", index=False)
outer_subject.to_csv(ARTIFACT_DIR / "V4_OOF_subject_scores.csv", index=False)
elastic_grid.to_csv(ARTIFACT_DIR / "V4_elasticnet_inner_grid.csv", index=False)
elastic_selection.to_csv(ARTIFACT_DIR / "V4_elasticnet_feature_selection_by_fold.csv", index=False)
pd.DataFrame(outer_fold_balance_rows).to_csv(
    ARTIFACT_DIR / "V4_outer_fold_balance_R2.csv", index=False
)
print("Nested tournament complete")
print("Outer fold balance:")
display(pd.DataFrame(outer_fold_balance_rows))

Nested tournament complete
Outer fold balance:


,repeat,fold,n_test_subjects,n_test_pairs,origin__PREEXISTING_414_SOURCE_REVIEW,origin__V4_TARGET_EXPANSION_2026_08_16,axis__COMPETITIVE,axis__PROJECT,axis__RECOGNITION,axis__STATUS
0,0,0,13,20,10,10,9,8,0,3
1,0,1,14,20,10,10,9,8,1,2
2,0,2,13,20,10,10,8,8,1,3
3,0,3,14,20,9,11,9,8,0,3
4,0,4,15,20,10,10,8,8,2,2
5,1,0,14,21,10,11,9,8,2,2
6,1,1,14,20,10,10,9,8,0,3
7,1,2,14,20,10,10,9,8,0,3
8,1,3,13,20,9,11,8,8,1,3
9,1,4,14,19,10,9,8,8,1,2


## 13. Overall leaderboard

In [16]:
repeat_metrics = (
    outer_subject.groupby(["model","repeat"])["correct"].mean()
    .reset_index(name="subject_macro")
)

leader_rows = []
for model, g in repeat_metrics.groupby("model"):
    vals = g.subject_macro.to_numpy(dtype=float)
    leader_rows.append({
        "model": model,
        "pairwise_mean": float(vals.mean()),
        "pairwise_std": float(vals.std(ddof=1)) if len(vals)>1 else 0.0,
        "pairwise_p10": float(np.quantile(vals,.10)),
        "pairwise_min": float(vals.min()),
    })
leaderboard = pd.DataFrame(leader_rows).sort_values("pairwise_mean", ascending=False).reset_index(drop=True)
leaderboard.to_csv(ARTIFACT_DIR / "V4_nested_leaderboard.csv", index=False)
display(leaderboard)

,model,pairwise_mean,pairwise_std,pairwise_p10,pairwise_min
0,NUISANCE_WITH_ORIGIN_DIAGNOSTIC,0.891304,0.000000,0.891304,0.891304
1,AGE_LATER_FIXED,0.547101,0.000000,0.547101,0.547101
2,ALL_ELASTICNET_AUTO,0.505072,0.225243,0.289372,0.253623
3,ALL_L2,0.480580,0.030889,0.457246,0.456280
4,TG10_STEM_BRANCH_L2,0.479082,0.042208,0.444203,0.439855
5,CONTROL_FIXED,0.476812,0.000000,0.476812,0.476812
6,NUISANCE_AGE_AXIS,0.475362,0.023162,0.452899,0.445652
7,TG10_STEM_L2,0.453720,0.067986,0.386957,0.344444
8,AGE_YOUNGER_FIXED,0.452899,0.000000,0.452899,0.452899
9,TG10_SB_RELATIONS_L2,0.444444,0.048808,0.392271,0.366184


## 14. Mandatory origin robustness

In [17]:
# Average repeated OOF correctness for each pair first, then subject-macro within origin.
pair_avg = (
    outer_pair.groupby(["model","pair_id","subject_id","pair_origin","axis"])["correct"]
    .mean().reset_index()
)

origin_rows=[]
for (model, origin), g in pair_avg.groupby(["model","pair_origin"]):
    subj = g.groupby("subject_id")["correct"].mean()
    origin_rows.append({
        "model":model,"pair_origin":origin,
        "n_pairs":len(g),"n_subjects":g.subject_id.nunique(),
        "subject_macro_pairwise":float(subj.mean()),
    })
origin_df=pd.DataFrame(origin_rows)
origin_df.to_csv(ARTIFACT_DIR / "V4_origin_robustness.csv",index=False)
display(origin_df.pivot(index="model",columns="pair_origin",values="subject_macro_pairwise"))

pair_origin,PREEXISTING_414_SOURCE_REVIEW,V4_TARGET_EXPANSION_2026-08-16
model,,
AGE_LATER_FIXED,0.163043,1.000000
AGE_YOUNGER_FIXED,0.836957,0.000000
ALL_ELASTICNET_AUTO,0.508696,0.487387
ALL_L2,0.478261,0.461982
CONTROL_FIXED,0.521739,0.407207
HGB_ALL_EXPLORATORY,0.436957,0.421802
NUISANCE_AGE_AXIS,0.686957,0.242342
NUISANCE_WITH_ORIGIN_DIAGNOSTIC,0.836957,1.000000
TG10_SB_RELATIONS_L2,0.460870,0.414414


## 15. Axis diagnostics (not a selection target)

In [18]:
axis_rows=[]
for (model, axis), g in pair_avg.groupby(["model","axis"]):
    subj=g.groupby("subject_id")["correct"].mean()
    axis_rows.append({
        "model":model,"axis":axis,"n_pairs":len(g),"n_subjects":g.subject_id.nunique(),
        "subject_macro_pairwise":float(subj.mean()),
    })
axis_df=pd.DataFrame(axis_rows)
axis_df.to_csv(ARTIFACT_DIR / "V4_axis_diagnostics.csv",index=False)
display(axis_df)

,model,axis,n_pairs,n_subjects,subject_macro_pairwise
0,AGE_LATER_FIXED,COMPETITIVE,43,26,0.625000
1,AGE_LATER_FIXED,PROJECT,40,33,0.540404
2,AGE_LATER_FIXED,RECOGNITION,4,3,0.500000
3,AGE_LATER_FIXED,STATUS,13,9,0.407407
4,AGE_YOUNGER_FIXED,COMPETITIVE,43,26,0.375000
5,AGE_YOUNGER_FIXED,PROJECT,40,33,0.459596
6,AGE_YOUNGER_FIXED,RECOGNITION,4,3,0.500000
7,AGE_YOUNGER_FIXED,STATUS,13,9,0.592593
8,ALL_ELASTICNET_AUTO,COMPETITIVE,43,26,0.554487
9,ALL_ELASTICNET_AUTO,PROJECT,40,33,0.481818


## 16. Paired subject bootstrap versus nuisance and Control

In [19]:
subject_avg = (
    outer_subject.groupby(["model","subject_id"])["correct"].mean().reset_index()
)
wide = subject_avg.pivot(index="subject_id",columns="model",values="correct")

rng=np.random.RandomState(SEED+999)
subject_ids=wide.index.to_numpy()

boot_rows=[]
for model in ASTRO_MODELS:
    if model not in wide.columns:
        continue
    for reference in ["NUISANCE_AGE_AXIS","CONTROL_FIXED"]:
        shared=wide[[model,reference]].dropna()
        ids=shared.index.to_numpy()
        deltas=[]
        for _ in range(N_BOOTSTRAP):
            sample=rng.choice(ids,size=len(ids),replace=True)
            d=(shared.loc[sample,model].to_numpy()-shared.loc[sample,reference].to_numpy()).mean()
            deltas.append(float(d))
        arr=np.asarray(deltas)
        observed=float((shared[model]-shared[reference]).mean())
        boot_rows.append({
            "model":model,"reference":reference,"observed_delta":observed,
            "bootstrap_mean_delta":float(arr.mean()),
            "ci025":float(np.quantile(arr,.025)),"ci975":float(np.quantile(arr,.975)),
            "p_delta_gt_0":float((arr>0).mean()),
        })
boot=pd.DataFrame(boot_rows)
boot.to_csv(ARTIFACT_DIR / "V4_model_reference_bootstrap.csv",index=False)
display(boot)

,model,reference,observed_delta,bootstrap_mean_delta,ci025,ci975,p_delta_gt_0
0,TG5_GROUP_L2,NUISANCE_AGE_AXIS,-0.052778,-0.052335,-0.172104,0.067150,0.19850
1,TG5_GROUP_L2,CONTROL_FIXED,-0.054227,-0.054156,-0.184420,0.074879,0.20700
2,TG10_STEM_L2,NUISANCE_AGE_AXIS,-0.021643,-0.021833,-0.145800,0.102899,0.36585
3,TG10_STEM_L2,CONTROL_FIXED,-0.023092,-0.022763,-0.158263,0.109278,0.37335
4,TG10_STEM_BRANCH_L2,NUISANCE_AGE_AXIS,0.003720,0.002876,-0.124928,0.127636,0.51835
5,TG10_STEM_BRANCH_L2,CONTROL_FIXED,0.002271,0.002052,-0.120099,0.123671,0.51415
6,TG10_SB_STRENGTH_L2,NUISANCE_AGE_AXIS,-0.049227,-0.048989,-0.170775,0.074880,0.21810
7,TG10_SB_STRENGTH_L2,CONTROL_FIXED,-0.050676,-0.051122,-0.163772,0.060435,0.18700
8,TG10_SB_RELATIONS_L2,NUISANCE_AGE_AXIS,-0.030918,-0.031619,-0.153623,0.091304,0.30450
9,TG10_SB_RELATIONS_L2,CONTROL_FIXED,-0.032367,-0.033199,-0.151691,0.083092,0.29145


## 17. Automatic ElasticNet feature-selection summary

In [20]:
if len(elastic_selection):
    selection_summary=(
        elastic_selection.groupby("feature")
        .agg(selection_frequency=("selected","mean"),mean_coefficient=("coefficient","mean"))
        .reset_index()
        .sort_values(["selection_frequency","feature"],ascending=[False,True])
    )
else:
    selection_summary=pd.DataFrame(columns=["feature","selection_frequency","mean_coefficient"])
selection_summary.to_csv(ARTIFACT_DIR / "V4_elasticnet_selection_frequency.csv",index=False)
display(selection_summary.head(100))

,feature,selection_frequency,mean_coefficient
200,diff__relation__sw_dw_branch_punish,0.40,0.095066
201,diff__relation__sw_dw_stem_clash,0.40,-0.071559
233,diff__shinsal__sw_세운_신살_길신__문창귀인_,0.40,-0.087537
342,diff__tgxstrength__sw_qi_sha_weak,0.40,-0.082125
369,diff__unseong__dw_state_절,0.40,0.040334
...,...,...,...
142,diff__basic__sw_branch_寅,0.08,-0.021664
162,diff__basic__sw_stem_癸,0.08,0.003364
164,diff__basic__sw_yy_yang,0.08,-0.001566
165,diff__basic__sw_yy_yin,0.08,0.001566


# 18. Predeclared V4 discovery decision

A model can be frozen only if all of the following hold:

1. winner-eligible architecture (HGB excluded at n=69)
2. overall repeated outer-CV subject-macro ≥ 0.55
3. ≥ +1%p versus `NUISANCE_AGE_AXIS`
4. bootstrap P(delta>0) ≥ 0.80 versus nuisance
5. ≥ +1%p versus production Control
6. bootstrap P(delta>0) ≥ 0.80 versus Control
7. subject-macro ≥0.50 in **each collection origin**
8. repeated-CV p10 ≥0.48

If none survive, the correct status is **no V4 architecture survives**. Do not open NEW_CONFIRM and do not tune the gate.

In [21]:
WINNER_ELIGIBLE = [
    "TG5_GROUP_L2","TG10_STEM_L2","TG10_STEM_BRANCH_L2",
    "TG10_SB_STRENGTH_L2","TG10_SB_RELATIONS_L2","ALL_L2","ALL_ELASTICNET_AUTO",
]

lead=leaderboard.set_index("model")
origin_pivot=origin_df.pivot(index="model",columns="pair_origin",values="subject_macro_pairwise")
boot_lookup=boot.set_index(["model","reference"])

gate_rows=[]
for model in WINNER_ELIGIBLE:
    overall=float(lead.loc[model,"pairwise_mean"])
    p10=float(lead.loc[model,"pairwise_p10"])
    d_n=float(boot_lookup.loc[(model,"NUISANCE_AGE_AXIS"),"observed_delta"])
    p_n=float(boot_lookup.loc[(model,"NUISANCE_AGE_AXIS"),"p_delta_gt_0"])
    d_c=float(boot_lookup.loc[(model,"CONTROL_FIXED"),"observed_delta"])
    p_c=float(boot_lookup.loc[(model,"CONTROL_FIXED"),"p_delta_gt_0"])
    origin_min=float(origin_pivot.loc[model].min())

    passes={
        "overall_ge_055":overall>=MIN_OVERALL_MACRO,
        "delta_nuisance_ge_001":d_n>=MIN_DELTA_VS_REFERENCE,
        "boot_nuisance_ge_080":p_n>=MIN_BOOT_P_DELTA_GT_0,
        "delta_control_ge_001":d_c>=MIN_DELTA_VS_REFERENCE,
        "boot_control_ge_080":p_c>=MIN_BOOT_P_DELTA_GT_0,
        "origin_min_ge_050":origin_min>=MIN_ORIGIN_MACRO,
        "p10_ge_048":p10>=MIN_P10,
    }
    gate_rows.append({
        "model":model,"overall":overall,"p10":p10,"delta_vs_nuisance":d_n,"p_vs_nuisance":p_n,
        "delta_vs_control":d_c,"p_vs_control":p_c,"origin_min":origin_min,
        **passes,"all_gates":all(passes.values()),
    })

gates=pd.DataFrame(gate_rows).sort_values(["all_gates","overall"],ascending=[False,False])
gates.to_csv(ARTIFACT_DIR / "V4_candidate_gates.csv",index=False)
display(gates)

survivors=gates[gates.all_gates]
if len(survivors):
    winner=survivors.iloc[0].model
    status="V4_DISCOVERY_ARCHITECTURE_FROZEN_FOR_NEW_HOLDOUT"
    reason="Best winner-eligible model passed overall, nuisance, Control, origin, and lower-tail gates."
else:
    winner=None
    status="V4_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE"
    reason="No astrology architecture passed all predeclared V4 discovery gates."

print(status, winner)

,model,overall,p10,delta_vs_nuisance,p_vs_nuisance,delta_vs_control,p_vs_control,origin_min,overall_ge_055,delta_nuisance_ge_001,boot_nuisance_ge_080,delta_control_ge_001,boot_control_ge_080,origin_min_ge_050,p10_ge_048,all_gates
6,ALL_ELASTICNET_AUTO,0.505072,0.289372,0.029710,0.71940,0.028261,0.68185,0.487387,False,True,False,True,False,False,False,False
5,ALL_L2,0.480580,0.457246,0.005217,0.53045,0.003768,0.52675,0.461982,False,False,False,False,False,False,False,False
2,TG10_STEM_BRANCH_L2,0.479082,0.444203,0.003720,0.51835,0.002271,0.51415,0.467391,False,False,False,False,False,False,False,False
1,TG10_STEM_L2,0.453720,0.386957,-0.021643,0.36585,-0.023092,0.37335,0.432609,False,False,False,False,False,False,False,False
4,TG10_SB_RELATIONS_L2,0.444444,0.392271,-0.030918,0.30450,-0.032367,0.29145,0.414414,False,False,False,False,False,False,False,False
3,TG10_SB_STRENGTH_L2,0.426135,0.384348,-0.049227,0.21810,-0.050676,0.18700,0.415217,False,False,False,False,False,False,False,False
0,TG5_GROUP_L2,0.422585,0.391087,-0.052778,0.19850,-0.054227,0.20700,0.411957,False,False,False,False,False,False,False,False


V4_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE None


## 19. Freeze / stop decision payload

In [22]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

payload={
    "version":"SAJU_ML_V4_TOURNAMENT_DECISION_V1",
    "notebook_version":NOTEBOOK_VERSION,
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":status,
    "winner":winner,
    "reason":reason,
    "target":{
        "n_pairs":int(len(pair_df)),"n_subjects":int(pair_df.subject_id.nunique()),
        "target_decision_sha256":sha256_file(DECISION_PATH),
        "pairs_sha256":sha256_file(PAIRS_PATH),
        "origin_guard_required":bool(ORIGIN_GUARD_REQUIRED),
    },
    "gates":gates.to_dict(orient="records"),
    "leaderboard":leaderboard.to_dict(orient="records"),
    "origin_robustness":origin_df.to_dict(orient="records"),
    "holdout_integrity":{
        "NEW_CONFIRM_loaded":False,"Validation_B_loaded":False,
        "Public_CHECK_loaded":False,"Public_FINAL_loaded":False,
    },
    "next_rule":(
        "If winner is frozen: do not tune on this target again; construct a completely new V4 holdout under one preregistered balanced collection protocol and score once. "
        "If no winner survives: do not open a holdout; diagnose target/feature representation or collect more development subjects."
    ),
}

with open(ARTIFACT_DIR/"V4_TOURNAMENT_DECISION.json","w",encoding="utf-8") as f:
    json.dump(payload,f,ensure_ascii=False,indent=2)
print(json.dumps({"status":status,"winner":winner,"next_rule":payload["next_rule"]},ensure_ascii=False,indent=2))

{
  "status": "V4_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE",
  "winner": null,
  "next_rule": "If winner is frozen: do not tune on this target again; construct a completely new V4 holdout under one preregistered balanced collection protocol and score once. If no winner survives: do not open a holdout; diagnose target/feature representation or collect more development subjects."
}


# What to send back

After **Restart Kernel → Run All**, send:

```text
research/ml/artifacts/v4_feature_model_tournament/
├── V4_TOURNAMENT_DECISION.json
├── V4_nested_leaderboard.csv
├── V4_origin_robustness.csv
├── V4_candidate_gates.csv
└── V4_model_reference_bootstrap.csv
```

The first four are sufficient for the next decision.

Do **not** open `NEW_CONFIRM` after this run. The new V4 target is not the legacy confirmation target.